# STimage-1K4M → HEST Pipeline

**Two parts:**
1. **HEST readers** — how Visium and Visium HD raw data gets converted to HESTData (from Tutorial 3, working code only)
2. **Overlap detection** — 6-feature scoring to find which STimage samples are already in HEST before converting

---

## Part 1A — Visium → HESTData

### What you need
| File | Source |
|------|--------|
| Full-res image (`.png` / `.tif`) | GEO supplemental or STimage HF |
| `filtered_feature_bc_matrix.h5` | Space Ranger output or GEO |
| `spatial/` folder (optional) | Space Ranger output |
| `.json` alignment file (optional) | Space Ranger output |

### Alignment strategy — use whichever applies (Step 1 → 2 → 3)

**Step 1:** `spatial/tissue_positions.csv` exists → pass `spatial_coord_path`  
**Step 2:** `.json` alignment file exists → pass `alignment_file_path`  
**Step 3:** Neither exists → autoalignment (needs ≥3 visible corner fiducials)

In [ ]:
# ── Download example Visium sample from GEO (Mac: curl instead of wget) ──
import os, gzip, shutil, urllib.request

os.makedirs('downloads', exist_ok=True)
BASE = 'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6215nnn/GSM6215674/suppl/'

files = {
    'GSM6215674_S13.png.gz'                       : 'GSM6215674%5FS13.png.gz',
    'GSM6215674_S13_filtered_feature_bc_matrix.h5': 'GSM6215674%5FS13%5Ffiltered%5Ffeature%5Fbc%5Fmatrix.h5',
}
for local, remote in files.items():
    dest = f'downloads/{local}'
    if not os.path.exists(dest) and not os.path.exists(dest.replace('.gz', '')):
        print(f'Downloading {local}...')
        urllib.request.urlretrieve(BASE + remote, dest)

gz = 'downloads/GSM6215674_S13.png.gz'
png = gz.replace('.gz', '')
if os.path.exists(gz) and not os.path.exists(png):
    with gzip.open(gz, 'rb') as fi, open(png, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    print('Unzipped PNG')

print('Files ready:', os.listdir('downloads'))

In [ ]:
from hest import VisiumReader

fullres_img_path = 'downloads/GSM6215674_S13.png'
bc_matrix_path   = 'downloads/GSM6215674_S13_filtered_feature_bc_matrix.h5'

# ── Step 3: autoalignment (no spatial/ folder, no .json) ──────────────────
st = VisiumReader().read(
    fullres_img_path,
    bc_matrix_path,
    save_autoalign=True   # saves a visualisation of fiducial detection
)
print(st)

In [ ]:
# ── Step 1: tissue_positions.csv already exists ───────────────────────────
# spatial_coord_path = 'path/to/spatial/'   # contains tissue_positions.csv
# st = VisiumReader().read(fullres_img_path, bc_matrix_path,
#                          spatial_coord_path=spatial_coord_path)

# ── Step 2: .json alignment file ─────────────────────────────────────────
# alignment_file_path = 'path/to/alignment.json'
# st = VisiumReader().read(fullres_img_path, bc_matrix_path,
#                          alignment_file_path=alignment_file_path)

# ── Inspect result ────────────────────────────────────────────────────────
print('AnnData:', st.adata)
print('Pixel size:', st.pixel_size, 'um/px')

In [ ]:
import os
os.makedirs('processed_visium', exist_ok=True)

# Fast save — adata + metadata + downscaled JPEG, no full WSI write
st.save('processed_visium', save_img=False)

# Spatial overlay plot
st.save_spatial_plot('processed_visium')

# Tissue segmentation (Otsu — fast, no GPU needed)
st.segment_tissue(method='otsu')
st.save_tissue_seg_pkl('processed_visium', 'tissue_seg_otsu')

print('Saved to processed_visium/')

## Part 1B — Visium HD → HESTData

### What you need (raw Space Ranger output)
| File / folder | Description |
|---------------|-------------|
| Full-res image (`.tif`) | CytAssist scan |
| `binned_outputs/square_016um/` | 16 µm bin counts + spatial coords |
| `metrics_summary.csv` (optional) | QC metrics |

> **Note:** STimage-1K4M VisiumHD samples are already processed (coord CSV + count CSV + PNG).  
> Use `VisiumHDReader` only when starting from raw Space Ranger output.

In [ ]:
from hest import VisiumHDReader

# ── Option A: auto-read a Space Ranger output directory ───────────────────
# visium_hd_dir = 'path/to/spaceranger_output/'
# st_hd = VisiumHDReader().auto_read(visium_hd_dir)

# ── Option B: explicit paths ──────────────────────────────────────────────
# st_hd = VisiumHDReader().read(
#     img_path       = 'path/to/image.tif',
#     square_16um_path = 'path/to/binned_outputs/square_016um/',
#     metrics_path   = 'path/to/metrics_summary.csv',   # optional
#     dst_bin_size_um = 128,   # pool 16µm bins into 128µm pseudo-Visium bins
# )

# ── Save (same API as Visium) ─────────────────────────────────────────────
# os.makedirs('processed_visium_hd', exist_ok=True)
# st_hd.save('processed_visium_hd', save_img=False)
# st_hd.save_spatial_plot('processed_visium_hd')

print('VisiumHDReader template ready — fill in paths above and uncomment.')

---
## Part 2 — STimage-1K4M Overlap Detection (6-feature scoring)

Identifies which STimage samples already exist in HEST before we convert them.

| # | STimage column | HEST column | Precision |
|---|---------------|-------------|-----------|
| 1 | `slide` → **GSM ID** | `download_page_link1` → GSM | Sample-level exact |
| 2 | `slide` → **GSE ID** | `download_page_link1` / `study_link` | Study-level exact |
| 3 | **`pmid`** | `study_link` → PMID | Study-level exact |
| 4 | **`title`** (paper title) | `dataset_title` | Fuzzy string match |
| 5 | **`tech`** | `st_technology` | Categorical filter |
| 6 | **`species`** + **`tissue`** + **`spot_num`** ±10% | `species` + `tissue` + `spots_under_tissue` | Composite fingerprint |

In [ ]:
import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher

STIMAGE_META = Path('../../STimage-1K4M/meta/meta_all_gene02122025.csv')
HEST_META_HF = 'hf://datasets/MahmoodLab/hest/HEST_v1_3_0.csv'

st_meta  = pd.read_csv(STIMAGE_META)
hest_meta = pd.read_csv(HEST_META_HF)

print(f'STimage-1K4M : {len(st_meta):,} samples')
print(f'HEST         : {len(hest_meta):,} samples')

In [ ]:
# ── Features 1 & 2: GSM and GSE IDs ──────────────────────────────────────
def extract_id(pattern, text):
    m = re.search(pattern, str(text))
    return m.group(1) if m else None

st_meta['gsm_id'] = st_meta['slide'].apply(lambda x: extract_id(r'(GSM\d+)', x))
st_meta['gse_id'] = st_meta['slide'].apply(lambda x: extract_id(r'(GSE\d+)', x))

hest_meta['gsm_id'] = hest_meta['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSM\d+)', x))
hest_meta['gse_id'] = (
    hest_meta['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x))
    .fillna(hest_meta['study_link'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x)))
)

hest_gsm_set = set(hest_meta['gsm_id'].dropna())
hest_gse_set = set(hest_meta['gse_id'].dropna())

st_meta['feat1_gsm'] = st_meta['gsm_id'].isin(hest_gsm_set)
st_meta['feat2_gse'] = st_meta['gse_id'].isin(hest_gse_set)

print(f'Feat1 GSM hits : {st_meta["feat1_gsm"].sum()}')
print(f'Feat2 GSE hits : {st_meta["feat2_gse"].sum()}')

In [ ]:
# ── Feature 3: PMID ───────────────────────────────────────────────────────
def pmid_set(val):
    if pd.isna(val): return set()
    return {p.strip() for p in str(val).split(',') if p.strip().isdigit()}

st_meta['pmid_set'] = st_meta['pmid'].apply(pmid_set)

hest_meta['pmid'] = hest_meta['study_link'].apply(
    lambda x: extract_id(r'pubmed\.ncbi\.nlm\.nih\.gov/(\d+)', x)
)
hest_pmid_set = set(hest_meta['pmid'].dropna())

st_meta['feat3_pmid'] = st_meta['pmid_set'].apply(lambda s: bool(s & hest_pmid_set))

shared_pmids = {p for s in st_meta[st_meta['feat3_pmid']]['pmid_set'] for p in s} & hest_pmid_set
print(f'Feat3 PMID hits : {st_meta["feat3_pmid"].sum()}  |  shared PMIDs: {sorted(shared_pmids)}')

In [ ]:
# ── Feature 4: Title fuzzy match ──────────────────────────────────────────
def normalise(text):
    if pd.isna(text): return ''
    return re.sub(r'[^a-z0-9 ]', '', str(text).lower())

hest_titles_norm = hest_meta['dataset_title'].apply(normalise).tolist()
TITLE_THRESHOLD = 0.6

def best_title_score(raw):
    s = normalise(raw)
    if not s: return 0.0
    return round(max(SequenceMatcher(None, s[:200], h[:200]).ratio() for h in hest_titles_norm), 3)

print('Computing title similarity (~1-2 min)...')
st_meta['feat4_title_score'] = st_meta['title'].apply(best_title_score)
st_meta['feat4_title']       = st_meta['feat4_title_score'] >= TITLE_THRESHOLD
print(f'Feat4 title hits (>= {TITLE_THRESHOLD}): {st_meta["feat4_title"].sum()}')

In [ ]:
# ── Feature 5: Technology compatibility ───────────────────────────────────
TECH_MAP = {'ST': 'Spatial Transcriptomics', 'Visium': 'Visium', 'VisiumHD': 'Visium HD'}
st_meta['tech_norm'] = st_meta['tech'].map(TECH_MAP)
hest_tech_set = set(hest_meta['st_technology'].dropna())

st_meta['feat5_tech'] = st_meta['tech_norm'].isin(hest_tech_set)
print('HEST technologies:', sorted(hest_tech_set))
print('STimage tech in HEST:', st_meta.groupby('tech')['feat5_tech'].sum().to_dict())

In [ ]:
# ── Feature 6: Composite fingerprint (species + tissue + spot count ±10%) ─
SPECIES_MAP = {
    'human': 'Homo sapiens', 'mouse': 'Mus musculus',
    'rattus norvegicus': 'Rattus norvegicus', 'pig': 'Sus scrofa',
    'human & mouse': None, 'fish': None, 'plant': None,
    'frog': None, 'ambystoma mexicanum': None,
}
st_meta['species_norm']    = st_meta['species'].str.lower().map(SPECIES_MAP)
st_meta['tissue_norm']     = st_meta['tissue'].str.lower().str.strip()
hest_meta['tissue_norm']   = hest_meta['tissue'].str.lower().str.strip()

SPOT_TOL = 0.10
fp_lookup = {}
for _, row in hest_meta.iterrows():
    key = (row.get('species', ''), row.get('tissue_norm', ''))
    if pd.notna(row.get('spots_under_tissue')):
        fp_lookup.setdefault(key, []).append(int(row['spots_under_tissue']))

def fingerprint_match(row):
    key = (row['species_norm'], row['tissue_norm'])
    if key not in fp_lookup or pd.isna(row['spot_num']): return False
    st_spots = int(row['spot_num'])
    return any(abs(st_spots - h) / max(st_spots, 1) <= SPOT_TOL for h in fp_lookup[key])

st_meta['feat6_fingerprint'] = st_meta.apply(fingerprint_match, axis=1)
print(f'Feat6 fingerprint hits: {st_meta["feat6_fingerprint"].sum()}')

In [ ]:
# ── Confidence scoring ────────────────────────────────────────────────────
WEIGHTS = {
    'feat1_gsm'        : 1.00,   # exact sample-level ID
    'feat2_gse'        : 0.60,   # same study
    'feat3_pmid'       : 0.60,   # same paper
    'feat4_title'      : 0.40,   # fuzzy title
    'feat5_tech'       : 0.10,   # tech compatibility
    'feat6_fingerprint': 0.30,   # species + tissue + spot count
}

st_meta['confidence'] = sum(st_meta[f].astype(float) * w for f, w in WEIGHTS.items())

def overlap_label(score):
    if score >= 1.00: return 'DEFINITE'
    if score >= 0.60: return 'LIKELY'
    if score >= 0.30: return 'POSSIBLE'
    return 'NOVEL'

st_meta['overlap_label'] = st_meta['confidence'].apply(overlap_label)

print('=== Label counts ===')
print(st_meta['overlap_label'].value_counts().to_string())
print('\n=== By technology ===')
print(st_meta.groupby(['overlap_label', 'tech']).size().unstack(fill_value=0).to_string())

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────
out_dir = Path('pipeline_output')
out_dir.mkdir(exist_ok=True)

feat_cols = [
    'slide','gsm_id','gse_id','pmid','tech','tissue','species','spot_num','gene_num',
    'feat1_gsm','feat2_gse','feat3_pmid','feat4_title','feat4_title_score',
    'feat5_tech','feat6_fingerprint','confidence','overlap_label'
]

st_meta[feat_cols].to_csv(out_dir / 'stimage_overlap_scored.csv', index=False)

novel    = st_meta[st_meta['overlap_label'] == 'NOVEL']
review   = st_meta[st_meta['overlap_label'] == 'POSSIBLE']
overlap  = st_meta[st_meta['overlap_label'].isin(['DEFINITE', 'LIKELY'])]

novel.to_csv(out_dir / 'stimage_novel.csv', index=False)
review.to_csv(out_dir / 'stimage_manual_review.csv', index=False)
overlap.to_csv(out_dir / 'stimage_confirmed_overlap.csv', index=False)

print(f'Outputs in {out_dir}/')
print(f'  stimage_overlap_scored.csv      — full scored table ({len(st_meta)} rows)')
print(f'  stimage_novel.csv               — {len(novel)} samples → convert to HEST')
print(f'  stimage_manual_review.csv       — {len(review)} samples → manual check')
print(f'  stimage_confirmed_overlap.csv   — {len(overlap)} confirmed duplicates → skip')

## Manual review guide

For each row in `stimage_manual_review.csv` (POSSIBLE label):
1. **GSE number** → search https://www.ncbi.nlm.nih.gov/geo/
2. **Title** → compare against HEST `dataset_title` in `stimage_overlap_scored.csv`
3. **PMID** → cross-check at https://pubmed.ncbi.nlm.nih.gov/

## Next: Notebook 8 — Convert `stimage_novel.csv` samples to HEST format

For each novel sample:
1. Download `{tech}/{id}_coord.csv`, `{tech}/{id}_count.csv`, `{tech}/{id}.png` from `jiawennnn/STimage-1K4M`
2. Build `AnnData` from count CSV + coord CSV
3. Convert PNG → pyramidal TIFF
4. Wrap in `HESTData(adata, wsi, pixel_size, meta)`
5. `st.save(out_dir, save_img=True, pyramidal=True)`